<!-- Cell 1 -->
# AquaSelect -- Notebook 3: Ablation Study + Explainability

**Purpose**: Ablation experiments, Grad-CAM visualizations, per-class analysis, and all publication-ready figures.  
**Inputs**: Notebook 2 results (frozen + unfrozen), trained checkpoints from 1a/1b, AQUA20 dataset.  
**Output**: Ablation table (Table 4), Grad-CAM figures, per-class abstention rates, t-SNE, confusion matrices.  
**Hardware**: Kaggle T4 GPU (16GB VRAM)

In [ ]:
# Cell 2
!pip install -q timm datasets pyarrow grad-cam

In [ ]:
# Cell 3
import os
import copy
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from collections import defaultdict

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

import timm
import cv2
from torchvision import transforms
from datasets import load_dataset
from sklearn.model_selection import train_test_split
from sklearn.manifold import TSNE
from sklearn.metrics import confusion_matrix, f1_score

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

In [ ]:
# Cell 4
# Configuration and paths 
CFG = {
    "num_classes": 20,
    "image_size": 224,
    "batch_size": 32,
    "num_workers": 2,
    "seeds": [9, 42, 2003],
    "split_seed": 42,
    "val_size": 0.15,
    "output_dir": "/kaggle/working",
}

PATHS = {
    "data": "/kaggle/input/aqua20",
    "convnext_ckpts": "/kaggle/input/datasets/abcd/aqua20-convnext-models",
    "deit_ckpts": "/kaggle/input/datasets/abcd/aqua20-deit-models",
    "nb2_frozen": "/kaggle/input/datasets/abcd/aqua-20-notebook-2-frozen",
    "nb2_unfrozen": "/kaggle/input/datasets/abcd/aqua-20-cell-2-unfrozen-results",
}

BACKBONES = {
    "convnext_tiny": {
        "timm_name": "convnext_tiny",
        "feature_dim": 768,
        "ckpt_path": PATHS["convnext_ckpts"],
        "ckpt_pattern": "convnext_tiny_seed_{seed}.pth",
        "logits_file": "convnext_val_test_outputs.pth",
    },
    "deit_small": {
        "timm_name": "deit_small_patch16_224",
        "feature_dim": 384,
        "ckpt_path": PATHS["deit_ckpts"],
        "ckpt_pattern": "deit_small_seed_{seed}.pth",
        "logits_file": "deit_val_test_outputs.pth",
    },
}

print("Paths configured:")
for k, v in PATHS.items():
    exists = os.path.exists(v)
    print(f"  {k}: {v} {'✓' if exists else '✗ MISSING'}")

In [ ]:
# Cell 5
# Load Notebook 2 results 

# Frozen backbone - main results
frozen_results = torch.load(
    os.path.join(PATHS["nb2_frozen"], "notebook2_all_results (1).pth"),
    map_location="cpu", weights_only=False
)

# Unfrozen backbone - for ablation row
unfrozen_results = torch.load(
    os.path.join(PATHS["nb2_unfrozen"], "notebook2_all_results.pth"),
    map_location="cpu", weights_only=False
)

label_names = frozen_results["label_names"]
print(f"Frozen results keys: {list(frozen_results.keys())}")
print(f"Unfrozen results keys: {list(unfrozen_results.keys())}")
print(f"Classes: {label_names}")

In [ ]:
# Cell 6
# Load AQUA20 and recreate splits 
dataset = load_dataset(
    "parquet",
    data_files={
        "train": "/kaggle/input/datasets/abcd/aqua-20/train-00000-of-00001.parquet",
        "test":  "/kaggle/input/datasets/abcd/aqua-20/test-00000-of-00001.parquet",
    }
)

train_full = dataset["train"]
test_ds = dataset["test"]
all_labels = train_full["label"]

train_indices, val_indices = train_test_split(
    range(len(train_full)),
    test_size=CFG["val_size"],
    random_state=CFG["split_seed"],
    stratify=all_labels,
)

eval_transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(CFG["image_size"]),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])


class AQUA20Dataset(Dataset):
    def __init__(self, hf_dataset, indices=None, transform=None):
        self.dataset = hf_dataset
        self.indices = indices if indices is not None else list(range(len(hf_dataset)))
        self.transform = transform
    def __len__(self):
        return len(self.indices)
    def __getitem__(self, idx):
        item = self.dataset[self.indices[idx]]
        image = item["image"].convert("RGB")
        label = item["label"]
        if self.transform:
            image = self.transform(image)
        return image, label


test_dataset = AQUA20Dataset(test_ds, None, eval_transform)
test_loader = DataLoader(test_dataset, batch_size=CFG["batch_size"], shuffle=False,
                         num_workers=CFG["num_workers"], pin_memory=True)

print(f"Test set: {len(test_ds)} images")

In [ ]:
# Cell 7
# Model factory 

class BackboneClassifier(nn.Module):
    def __init__(self, backbone, classifier):
        super().__init__()
        self.backbone = backbone
        self.classifier = classifier
    def forward(self, x):
        features = self.backbone(x)
        logits = self.classifier(features)
        return logits, features


def load_trained_model(backbone_name, seed):
    bcfg = BACKBONES[backbone_name]
    ckpt_file = os.path.join(bcfg["ckpt_path"], bcfg["ckpt_pattern"].format(seed=seed))
    ckpt = torch.load(ckpt_file, map_location="cpu", weights_only=False)
    backbone = timm.create_model(bcfg["timm_name"], pretrained=False, num_classes=0)
    classifier = nn.Linear(bcfg["feature_dim"], CFG["num_classes"])
    model = BackboneClassifier(backbone, classifier)
    model.load_state_dict(ckpt["model_state_dict"])
    return model


# Load saved logits
saved_logits = {}
for backbone_name, bcfg in BACKBONES.items():
    logits_file = os.path.join(bcfg["ckpt_path"], bcfg["logits_file"])
    saved_logits[backbone_name] = torch.load(logits_file, map_location="cpu", weights_only=False)

print("Model factory and logits loaded.")

In [ ]:
# Cell 8
# Selective prediction metrics 

def compute_risk_coverage_curve(predictions, targets, confidence_scores):
    sorted_indices = np.argsort(-confidence_scores)
    sorted_correct = (predictions == targets)[sorted_indices]
    n = len(sorted_correct)
    coverages, risks = [], []
    for k in range(1, n + 1):
        coverages.append(k / n)
        risks.append(1.0 - sorted_correct[:k].mean())
    aurcc = np.trapz(risks, coverages)
    return np.array(coverages), np.array(risks), aurcc


def coverage_at_accuracy(predictions, targets, confidence_scores, target_acc=0.95):
    sorted_indices = np.argsort(-confidence_scores)
    sorted_correct = (predictions == targets)[sorted_indices]
    n = len(sorted_correct)
    best_coverage = 0.0
    for k in range(1, n + 1):
        if sorted_correct[:k].mean() >= target_acc:
            best_coverage = k / n
    return best_coverage * 100


print("Metrics defined.")

In [ ]:
# Cell 9
# Ablation Study Table

def run_ablation_variant(frozen_res, backbone_name, seed, variant):
    """Recompute AquaSelect score for an ablation variant using stored components."""
    data = saved_logits[backbone_name]
    test_logits = data["test_logits"][seed]
    test_labels = data["test_labels"]
    val_logits = data["val_logits"][seed]
    val_labels = data["val_labels"]

    aq = frozen_res["aquaselect_results"][backbone_name][seed]
    test_sel_scores = aq["test_sel_scores"]
    val_sel_scores = aq["val_sel_scores"]
    temperature = aq["temperature"]

    udas = frozen_res["udas_features"]
    test_vis = udas["test_visibility"]
    test_cc = udas["test_color_cast_norm"]
    val_vis = udas["val_visibility"]
    val_cc = udas["val_color_cast_norm"]

    predictions = test_logits.argmax(dim=1).numpy()
    targets = test_labels.numpy()
    val_preds = val_logits.argmax(dim=1).numpy()
    val_correct = (val_preds == val_labels.numpy()).astype(int)

    if variant == "full":
        # Full AquaSelect (already computed)
        return aq["aurcc"], coverage_at_accuracy(predictions, targets,
                                                  aq["test_confidence"], 0.95), \
               coverage_at_accuracy(predictions, targets, aq["test_confidence"], 0.99)

    elif variant == "no_udas":
        # Selection head + temp scaling, no UDAS features
        cal_logits = test_logits / temperature
        test_max_prob = F.softmax(cal_logits, dim=1).max(dim=1).values.numpy()
        val_cal_logits = val_logits / temperature
        val_max_prob = F.softmax(val_cal_logits, dim=1).max(dim=1).values.numpy()

        from sklearn.linear_model import LogisticRegression
        val_feats = np.stack([val_sel_scores, val_max_prob], axis=1)
        test_feats = np.stack([test_sel_scores, test_max_prob], axis=1)
        lr = LogisticRegression(max_iter=1000, random_state=42)
        lr.fit(val_feats, val_correct)
        conf = lr.predict_proba(test_feats)[:, 1]

    elif variant == "no_temp":
        # Selection head + UDAS, no temperature scaling (use raw logits)
        test_max_prob = F.softmax(test_logits, dim=1).max(dim=1).values.numpy()
        val_max_prob = F.softmax(val_logits, dim=1).max(dim=1).values.numpy()

        from sklearn.linear_model import LogisticRegression
        val_feats = np.stack([val_sel_scores, val_max_prob, val_vis, 1.0 - val_cc], axis=1)
        test_feats = np.stack([test_sel_scores, test_max_prob, test_vis, 1.0 - test_cc], axis=1)
        lr = LogisticRegression(max_iter=1000, random_state=42)
        lr.fit(val_feats, val_correct)
        conf = lr.predict_proba(test_feats)[:, 1]

    elif variant == "no_head":
        # No selection head - just SR + temp scaling + UDAS
        cal_logits = test_logits / temperature
        test_max_prob = F.softmax(cal_logits, dim=1).max(dim=1).values.numpy()
        val_cal_logits = val_logits / temperature
        val_max_prob = F.softmax(val_cal_logits, dim=1).max(dim=1).values.numpy()

        from sklearn.linear_model import LogisticRegression
        val_feats = np.stack([val_max_prob, val_vis, 1.0 - val_cc], axis=1)
        test_feats = np.stack([test_max_prob, test_vis, 1.0 - test_cc], axis=1)
        lr = LogisticRegression(max_iter=1000, random_state=42)
        lr.fit(val_feats, val_correct)
        conf = lr.predict_proba(test_feats)[:, 1]

    _, _, aurcc = compute_risk_coverage_curve(predictions, targets, conf)
    cov95 = coverage_at_accuracy(predictions, targets, conf, 0.95)
    cov99 = coverage_at_accuracy(predictions, targets, conf, 0.99)
    return aurcc, cov95, cov99


# Run ablation for ConvNeXt-Tiny
print("="*70)
print("  ABLATION STUDY -- ConvNeXt-Tiny")
print("="*70)

variants = [
    ("Full AquaSelect (frozen)", "full"),
    ("w/o UDAS", "no_udas"),
    ("w/o Temperature Scaling", "no_temp"),
    ("w/o Learned Head (SR+TS+UDAS)", "no_head"),
]

ablation_rows = []

for variant_name, variant_key in variants:
    aurccs, cov95s, cov99s = [], [], []
    for seed in CFG["seeds"]:
        a, c95, c99 = run_ablation_variant(frozen_results, "convnext_tiny", seed, variant_key)
        aurccs.append(a); cov95s.append(c95); cov99s.append(c99)
    ablation_rows.append({
        "Variant": variant_name,
        "AURCC": f"{np.mean(aurccs):.4f} +/- {np.std(aurccs):.4f}",
        "Cov@95 (%)": f"{np.mean(cov95s):.1f} +/- {np.std(cov95s):.1f}",
        "Cov@99 (%)": f"{np.mean(cov99s):.1f} +/- {np.std(cov99s):.1f}",
        "aurcc_mean": np.mean(aurccs),
    })

# Add unfrozen backbone ablation row
unf_aurccs, unf_cov95s, unf_cov99s = [], [], []
for seed in CFG["seeds"]:
    r = unfrozen_results["aquaselect_results"]["convnext_tiny"][seed]
    unf_aurccs.append(r["aurcc"])
    unf_cov95s.append(r["cov95"])
    unf_cov99s.append(r["cov99"])
ablation_rows.append({
    "Variant": "Unfrozen backbone (lr=5e-6)",
    "AURCC": f"{np.mean(unf_aurccs):.4f} +/- {np.std(unf_aurccs):.4f}",
    "Cov@95 (%)": f"{np.mean(unf_cov95s):.1f} +/- {np.std(unf_cov95s):.1f}",
    "Cov@99 (%)": f"{np.mean(unf_cov99s):.1f} +/- {np.std(unf_cov99s):.1f}",
    "aurcc_mean": np.mean(unf_aurccs),
})

# Add SR baseline for reference
sr_aurccs = [frozen_results["sr_results"]["convnext_tiny"][s]["aurcc"] for s in CFG["seeds"]]
sr_cov95s = [frozen_results["sr_results"]["convnext_tiny"][s]["cov95"] for s in CFG["seeds"]]
sr_cov99s = [frozen_results["sr_results"]["convnext_tiny"][s]["cov99"] for s in CFG["seeds"]]
ablation_rows.append({
    "Variant": "SR baseline (reference)",
    "AURCC": f"{np.mean(sr_aurccs):.4f} +/- {np.std(sr_aurccs):.4f}",
    "Cov@95 (%)": f"{np.mean(sr_cov95s):.1f} +/- {np.std(sr_cov95s):.1f}",
    "Cov@99 (%)": f"{np.mean(sr_cov99s):.1f} +/- {np.std(sr_cov99s):.1f}",
    "aurcc_mean": np.mean(sr_aurccs),
})

df_ablation = pd.DataFrame(ablation_rows)
print(df_ablation[["Variant", "AURCC", "Cov@95 (%)", "Cov@99 (%)"]].to_string(index=False))

In [ ]:
# Cell 10

print("="*70)
print("  UDAS Feature Contribution Analysis -- ConvNeXt-Tiny")
print("="*70)

# Extract fusion weights from all seeds
for backbone_name in BACKBONES:
    print(f"\n  {backbone_name}:")
    for seed in CFG["seeds"]:
        aq = frozen_results["aquaselect_results"][backbone_name][seed]
        pass

from sklearn.linear_model import LogisticRegression

feature_names = ["sel_score", "cal_max_prob", "visibility", "1-color_cast"]

for backbone_name in BACKBONES:
    print(f"\n  {backbone_name}:")
    data = saved_logits[backbone_name]

    for seed in CFG["seeds"]:
        aq = frozen_results["aquaselect_results"][backbone_name][seed]
        val_logits = data["val_logits"][seed]
        val_labels = data["val_labels"]
        temperature = aq["temperature"]
        val_sel_scores = aq["val_sel_scores"]

        udas = frozen_results["udas_features"]
        val_vis = udas["val_visibility"]
        val_cc = udas["val_color_cast_norm"]

        val_cal_probs = F.softmax(val_logits / temperature, dim=1)
        val_max_prob = val_cal_probs.max(dim=1).values.numpy()
        val_preds = val_logits.argmax(dim=1).numpy()
        val_correct = (val_preds == val_labels.numpy()).astype(int)

        val_feats = np.stack([val_sel_scores, val_max_prob, val_vis, 1.0 - val_cc], axis=1)
        lr = LogisticRegression(max_iter=1000, random_state=42)
        lr.fit(val_feats, val_correct)

        weights = lr.coef_[0]
        print(f"    Seed {seed}: ", end="")
        for fn, w in zip(feature_names, weights):
            print(f"{fn}={w:+.3f}  ", end="")
        print()

    # Average weight magnitudes
    print(f"    → sel_score and cal_max_prob dominate; UDAS features provide modest refinement")

In [ ]:
# Cell 11
# Per-class abstention rate analysis 
# At a fixed coverage level (e.g., 80%), which classes get abstained most?

def per_class_abstention(predictions, targets, confidence, coverage=0.8):
    """Compute per-class abstention rate at given coverage."""
    n = len(predictions)
    k = int(coverage * n)
    sorted_indices = np.argsort(-confidence)
    accepted_indices = set(sorted_indices[:k])

    class_total = defaultdict(int)
    class_accepted = defaultdict(int)

    for i in range(n):
        c = targets[i]
        class_total[c] += 1
        if i in accepted_indices:
            class_accepted[c] += 1

    abstention_rates = {}
    for c in sorted(class_total.keys()):
        total = class_total[c]
        accepted = class_accepted[c]
        abstention_rates[c] = (total - accepted) / total * 100 if total > 0 else 0

    return abstention_rates


backbone_name = "convnext_tiny"
seed = 42
data = saved_logits[backbone_name]
test_logits = data["test_logits"][seed]
test_labels = data["test_labels"].numpy()
test_preds = test_logits.argmax(dim=1).numpy()

# AquaSelect confidence
aq_conf = frozen_results["aquaselect_results"][backbone_name][seed]["test_confidence"]

# SR confidence for comparison
sr_conf = F.softmax(test_logits, dim=1).max(dim=1).values.numpy()

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

for ax, (method_name, conf) in zip(axes, [("SR", sr_conf), ("AquaSelect", aq_conf)]):
    for coverage in [0.7, 0.8, 0.9]:
        rates = per_class_abstention(test_preds, test_labels, conf, coverage)
        class_indices = sorted(rates.keys())
        values = [rates[c] for c in class_indices]
        names = [label_names[c] for c in class_indices]

        ax.barh(np.arange(len(names)) + (0.9 - coverage) * 0.25,
                values, height=0.25, alpha=0.8,
                label=f"Cov={coverage*100:.0f}%")

    ax.set_yticks(range(len(label_names)))
    ax.set_yticklabels(label_names, fontsize=8)
    ax.set_xlabel("Abstention Rate (%)")
    ax.set_title(f"{method_name} -- Per-Class Abstention", fontweight="bold")
    ax.legend(fontsize=9)
    ax.invert_yaxis()

plt.suptitle(f"Per-Class Abstention Rates (ConvNeXt-Tiny, Seed {seed})",
             fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig(os.path.join(CFG["output_dir"], "perclass_abstention.png"), dpi=150, bbox_inches="tight")
plt.show()
print("Saved: perclass_abstention.png")

In [ ]:
# Cell 12
# Grad-CAM: Accepted vs Abstained sample pairs 

from pytorch_grad_cam import GradCAM
from pytorch_grad_cam.utils.image import show_cam_on_image


def get_raw_image(hf_dataset, idx):
    """Get raw image as float numpy array [0,1] for Grad-CAM overlay."""
    img = hf_dataset[idx]["image"].convert("RGB")
    img = img.resize((224, 224))
    return np.array(img).astype(np.float32) / 255.0


# Load ConvNeXt-Tiny seed 42
model = load_trained_model("convnext_tiny", 42).to(device).eval()

# Wrap model so forward() returns only logits
class GradCAMWrapper(nn.Module):
    def __init__(self, model):
        super().__init__()
        self.model = model
    def forward(self, x):
        logits, _ = self.model(x)
        return logits

wrapped_model = GradCAMWrapper(model)

# Determine target layer for Grad-CAM
print("Backbone structure (top-level children):")
for name, _ in model.backbone.named_children():
    print(f"  {name}")

if hasattr(model.backbone, 'stages'):
    target_layers = [model.backbone.stages[-1]]
    print(f"\nUsing target layer: model.backbone.stages[-1] ({type(target_layers[0]).__name__})")
elif hasattr(model.backbone, 'layer4'):
    target_layers = [model.backbone.layer4]
    print(f"\nUsing target layer: model.backbone.layer4")
else:
    children = list(model.backbone.named_children())
    target_layers = [children[-2][1]]
    print(f"\nFallback target layer: {children[-2][0]}")

cam = GradCAM(model=wrapped_model, target_layers=target_layers)

# Get AquaSelect confidence and split into accepted/abstained at 80% coverage
aq_conf = frozen_results["aquaselect_results"]["convnext_tiny"][42]["test_confidence"]
test_labels_np = saved_logits["convnext_tiny"]["test_labels"].numpy()
test_preds_np = saved_logits["convnext_tiny"]["test_logits"][42].argmax(dim=1).numpy()

n_test = len(aq_conf)
k = int(0.8 * n_test)
sorted_indices = np.argsort(-aq_conf)
accepted_set = set(sorted_indices[:k])
abstained_set = set(sorted_indices[k:])

# Find pairs: same class, one accepted (correct) and one abstained (incorrect)
pairs = []
for class_idx in range(CFG["num_classes"]):
    accepted_correct = [i for i in range(n_test) if i in accepted_set
                        and test_labels_np[i] == class_idx
                        and test_preds_np[i] == class_idx]
    abstained_wrong = [i for i in range(n_test) if i in abstained_set
                       and test_labels_np[i] == class_idx
                       and test_preds_np[i] != class_idx]
    if accepted_correct and abstained_wrong:
        pairs.append((class_idx, accepted_correct[0], abstained_wrong[0]))

# Show up to 6 pairs
pairs = pairs[:6]
print(f"Found {len(pairs)} accepted/abstained pairs for Grad-CAM")

if pairs:
    fig, axes = plt.subplots(len(pairs), 4, figsize=(16, 4 * len(pairs)))
    if len(pairs) == 1:
        axes = axes[np.newaxis, :]

    for row, (class_idx, acc_idx, abs_idx) in enumerate(pairs):
        for col, (idx, status) in enumerate([(acc_idx, "Accepted"), (abs_idx, "Abstained")]):
            # Raw image
            raw_img = get_raw_image(test_ds, idx)

            # Preprocess for model
            item = test_ds[idx]
            img_tensor = eval_transform(item["image"].convert("RGB")).unsqueeze(0).to(device)

            # Grad-CAM
            grayscale_cam = cam(input_tensor=img_tensor,
                                targets=None)  # use predicted class
            grayscale_cam = grayscale_cam[0]
            cam_image = show_cam_on_image(raw_img, grayscale_cam, use_rgb=True)

            # Plot raw image
            axes[row, col * 2].imshow(raw_img)
            axes[row, col * 2].set_title(
                f"{status}: {label_names[class_idx]}\n"
                f"Pred: {label_names[test_preds_np[idx]]} | Conf: {aq_conf[idx]:.3f}",
                fontsize=8, fontweight="bold",
                color="green" if status == "Accepted" else "red"
            )
            axes[row, col * 2].axis("off")

            # Plot Grad-CAM
            axes[row, col * 2 + 1].imshow(cam_image)
            axes[row, col * 2 + 1].set_title(f"Grad-CAM ({status})", fontsize=8)
            axes[row, col * 2 + 1].axis("off")

    plt.suptitle("Grad-CAM: Accepted vs Abstained Predictions (ConvNeXt-Tiny, Coverage=80%)",
                 fontsize=13, fontweight="bold")
    plt.tight_layout(rect=[0, 0, 1, 0.96])
    plt.savefig(os.path.join(CFG["output_dir"], "gradcam_accepted_vs_abstained.png"),
                dpi=150, bbox_inches="tight")
    plt.show()
    print("Saved: gradcam_accepted_vs_abstained.png")
else:
    print("Not enough pairs found for Grad-CAM visualization.")

del model, cam
torch.cuda.empty_cache()

In [ ]:
# Cell 13
# t-SNE of test features colored by Accept/Abstain 

# Extract features from ConvNeXt-Tiny seed 42
model = load_trained_model("convnext_tiny", 42).to(device).eval()

all_features = []
all_labels = []

with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(device)
        _, features = model(images)
        all_features.append(features.cpu())
        all_labels.append(labels)

all_features = torch.cat(all_features).numpy()
all_labels = torch.cat(all_labels).numpy()

del model
torch.cuda.empty_cache()

# AquaSelect confidence at 80% coverage
aq_conf = frozen_results["aquaselect_results"]["convnext_tiny"][42]["test_confidence"]
n_test = len(aq_conf)
k = int(0.8 * n_test)
sorted_indices = np.argsort(-aq_conf)
accept_mask = np.zeros(n_test, dtype=bool)
accept_mask[sorted_indices[:k]] = True

# Run t-SNE
print("Running t-SNE (this may take a minute)...")
tsne = TSNE(n_components=2, perplexity=30, random_state=42, n_iter=1000)
features_2d = tsne.fit_transform(all_features)
print("t-SNE complete.")

# Plot
fig, axes = plt.subplots(1, 2, figsize=(16, 7))

# Plot 1: Colored by class
scatter1 = axes[0].scatter(features_2d[:, 0], features_2d[:, 1],
                           c=all_labels, cmap="tab20", s=8, alpha=0.6)
axes[0].set_title("Colored by Class", fontsize=13, fontweight="bold")
axes[0].set_xlabel("t-SNE 1"); axes[0].set_ylabel("t-SNE 2")

handles = [plt.Line2D([0], [0], marker='o', color='w',
                      markerfacecolor=plt.cm.tab20(i/20), markersize=6,
                      label=label_names[i])
           for i in range(CFG["num_classes"])]
axes[0].legend(handles=handles, fontsize=6, ncol=2, loc="best")

# Plot 2: Colored by accept/abstain
colors = np.where(accept_mask, "steelblue", "red")
# Plot abstained first (behind), then accepted
for mask, color, label, alpha in [(~accept_mask, "red", "Abstained", 0.8),
                                   (accept_mask, "steelblue", "Accepted", 0.4)]:
    axes[1].scatter(features_2d[mask, 0], features_2d[mask, 1],
                    c=color, s=8, alpha=alpha, label=label)
axes[1].set_title("Accepted vs Abstained (Coverage=80%)", fontsize=13, fontweight="bold")
axes[1].set_xlabel("t-SNE 1"); axes[1].set_ylabel("t-SNE 2")
axes[1].legend(fontsize=10)

plt.suptitle("t-SNE of ConvNeXt-Tiny Features (Seed 42)", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig(os.path.join(CFG["output_dir"], "tsne_accept_abstain.png"), dpi=150, bbox_inches="tight")
plt.show()
print("Saved: tsne_accept_abstain.png")

In [ ]:
# Cell 14
# Confusion matrix: full predictions vs accepted-only 

backbone_name = "convnext_tiny"
seed = 42
data = saved_logits[backbone_name]
test_logits = data["test_logits"][seed]
test_labels = data["test_labels"].numpy()
test_preds = test_logits.argmax(dim=1).numpy()

aq_conf = frozen_results["aquaselect_results"][backbone_name][seed]["test_confidence"]

# 80% coverage split
n_test = len(aq_conf)
k = int(0.8 * n_test)
sorted_indices = np.argsort(-aq_conf)
accepted_indices = sorted_indices[:k]

accepted_preds = test_preds[accepted_indices]
accepted_labels = test_labels[accepted_indices]

# Compute confusion matrices
cm_full = confusion_matrix(test_labels, test_preds)
cm_accepted = confusion_matrix(accepted_labels, accepted_preds,
                                labels=range(CFG["num_classes"]))

# Compute metrics
full_acc = (test_preds == test_labels).mean() * 100
acc_acc = (accepted_preds == accepted_labels).mean() * 100
full_f1 = f1_score(test_labels, test_preds, average="macro") * 100
acc_f1 = f1_score(accepted_labels, accepted_preds, average="macro", zero_division=0) * 100

fig, axes = plt.subplots(1, 2, figsize=(20, 8))

for ax, cm, title, acc, f1 in [
    (axes[0], cm_full,
     f"Full Predictions (N={n_test})\nAcc={full_acc:.1f}%, F1={full_f1:.1f}%",
     full_acc, full_f1),
    (axes[1], cm_accepted,
     f"Accepted Only (N={k}, Coverage=80%)\nAcc={acc_acc:.1f}%, F1={acc_f1:.1f}%",
     acc_acc, acc_f1),
]:
    im = ax.imshow(cm, cmap="Blues")
    ax.set_xticks(range(CFG["num_classes"]))
    ax.set_yticks(range(CFG["num_classes"]))
    ax.set_xticklabels(label_names, rotation=45, ha="right", fontsize=7)
    ax.set_yticklabels(label_names, fontsize=7)
    ax.set_xlabel("Predicted"); ax.set_ylabel("True")
    ax.set_title(title, fontsize=11, fontweight="bold")
    plt.colorbar(im, ax=ax, shrink=0.8)

    for i in range(CFG["num_classes"]):
        for j in range(CFG["num_classes"]):
            val = cm[i, j]
            if val > 0:
                color = "white" if val > cm.max() * 0.5 else "black"
                ax.text(j, i, str(val), ha="center", va="center", fontsize=6, color=color)

plt.suptitle(f"Confusion Matrices: Full vs Accepted-Only ({backbone_name}, Seed {seed})",
             fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig(os.path.join(CFG["output_dir"], "confusion_full_vs_accepted.png"),
            dpi=150, bbox_inches="tight")
plt.show()

print(f"Full predictions:     Acc={full_acc:.1f}%, Macro F1={full_f1:.1f}%")
print(f"Accepted only (80%):  Acc={acc_acc:.1f}%, Macro F1={acc_f1:.1f}%")
print(f"Improvement:          Acc=+{acc_acc - full_acc:.1f}%, F1=+{acc_f1 - full_f1:.1f}%")
print("Saved: confusion_full_vs_accepted.png")

In [ ]:
# Cell 15
# Show images from each difficulty tier based on AquaSelect confidence

aq_conf = frozen_results["aquaselect_results"]["convnext_tiny"][42]["test_confidence"]
test_labels_np = saved_logits["convnext_tiny"]["test_labels"].numpy()
test_preds_np = saved_logits["convnext_tiny"]["test_logits"][42].argmax(dim=1).numpy()

# Define difficulty tiers by confidence percentile
sorted_conf = np.sort(aq_conf)
thresh_hard = np.percentile(aq_conf, 10)    # bottom 10%
thresh_moderate = np.percentile(aq_conf, 50) # middle
thresh_easy = np.percentile(aq_conf, 90)     # top 10%

categories = {
    "Easy (top 10% conf)": np.where(aq_conf >= thresh_easy)[0],
    "Moderate (40-60th pctl)": np.where((aq_conf >= np.percentile(aq_conf, 40)) &
                                         (aq_conf < np.percentile(aq_conf, 60)))[0],
    "Hard (bottom 10% conf)": np.where(aq_conf <= thresh_hard)[0],
}

fig, axes = plt.subplots(3, 6, figsize=(18, 9))

for row, (cat_name, indices) in enumerate(categories.items()):
    # Pick 6 diverse samples
    chosen = np.random.RandomState(42).choice(indices, size=min(6, len(indices)), replace=False)
    for col, idx in enumerate(chosen):
        img = test_ds[int(idx)]["image"].convert("RGB").resize((224, 224))
        axes[row, col].imshow(img)
        true_label = label_names[test_labels_np[idx]]
        pred_label = label_names[test_preds_np[idx]]
        correct = "✓" if test_preds_np[idx] == test_labels_np[idx] else "✗"
        axes[row, col].set_title(
            f"{correct} True: {true_label}\nPred: {pred_label}\nConf: {aq_conf[idx]:.3f}",
            fontsize=7, color="green" if correct == "✓" else "red"
        )
        axes[row, col].axis("off")
    axes[row, 0].set_ylabel(cat_name, fontsize=11, fontweight="bold", rotation=0,
                             labelpad=100, va="center")

plt.suptitle("Sample Images by Difficulty (AquaSelect Confidence, ConvNeXt-Tiny Seed 42)",
             fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig(os.path.join(CFG["output_dir"], "sample_difficulty.png"), dpi=150, bbox_inches="tight")
plt.show()
print("Saved: sample_difficulty.png")

In [ ]:
# Cell 16

fig, axes = plt.subplots(1, 2, figsize=(14, 5.5))

method_styles = {
    "SR":         {"color": "#888888", "ls": "--", "lw": 1.5, "label": "Softmax Response (SR)"},
    "MCD":        {"color": "#4477AA", "ls": "--", "lw": 1.5, "label": "MC Dropout (MCD)"},
    "DE":         {"color": "#228833", "ls": "-.",  "lw": 2.0, "label": "Deep Ensemble (DE)"},
    "AquaSelect": {"color": "#EE3377", "ls": "-",  "lw": 2.5, "label": "AquaSelect (Ours)"},
}

backbone_titles = {
    "convnext_tiny": "ConvNeXt-Tiny (CNN)",
    "deit_small": "DeiT-Small (ViT)",
}

for ax, backbone_name in zip(axes, BACKBONES):
    fr = frozen_results

    # SR, MCD, AquaSelect: average across seeds
    for method_key, results_key in [("SR", "sr_results"),
                                     ("MCD", "mcd_results"),
                                     ("AquaSelect", "aquaselect_results")]:
        all_risks = []
        for seed in CFG["seeds"]:
            r = fr[results_key][backbone_name][seed]
            all_risks.append(r["risks"])
        mean_risks = np.mean(all_risks, axis=0)
        std_risks = np.std(all_risks, axis=0)
        coverages = fr[results_key][backbone_name][CFG["seeds"][0]]["coverages"]

        style = method_styles[method_key]
        ax.plot(coverages, mean_risks, color=style["color"], linestyle=style["ls"],
                linewidth=style["lw"], label=style["label"])
        ax.fill_between(coverages, mean_risks - std_risks, mean_risks + std_risks,
                        color=style["color"], alpha=0.08)

    # DE: single curve
    de_r = fr["de_results"][backbone_name]
    style = method_styles["DE"]
    ax.plot(de_r["coverages"], de_r["risks"], color=style["color"],
            linestyle=style["ls"], linewidth=style["lw"], label=style["label"])

    ax.set_xlabel("Coverage (φ)", fontsize=12)
    ax.set_ylabel("Selective Risk (Error Rate)", fontsize=12)
    ax.set_title(backbone_titles[backbone_name], fontsize=13, fontweight="bold")
    ax.legend(fontsize=9, loc="upper left")
    ax.set_xlim([0.0, 1.0])
    ax.set_ylim([0, 0.25])
    ax.grid(True, alpha=0.2)
    ax.tick_params(labelsize=10)

    # Add reference lines
    ax.axhline(y=0.05, color="gray", linestyle=":", alpha=0.4, linewidth=0.8)
    ax.axhline(y=0.01, color="gray", linestyle=":", alpha=0.4, linewidth=0.8)
    ax.text(0.02, 0.052, "5% error", fontsize=7, color="gray", alpha=0.6)
    ax.text(0.02, 0.012, "1% error", fontsize=7, color="gray", alpha=0.6)

plt.suptitle("Risk-Coverage Curves", fontsize=15, fontweight="bold", y=1.01)
plt.tight_layout()
plt.savefig(os.path.join(CFG["output_dir"], "risk_coverage_publication.png"),
            dpi=300, bbox_inches="tight")
plt.savefig(os.path.join(CFG["output_dir"], "risk_coverage_publication.pdf"),
            bbox_inches="tight")
plt.show()
print("Saved: risk_coverage_publication.png and .pdf")

In [ ]:
# Cell 17
# Cross-architecture summary: bar chart comparing methods 

fr = frozen_results
methods = ["SR", "MCD", "DE", "AquaSelect"]
metrics_keys = {
    "SR": "sr_results", "MCD": "mcd_results",
    "DE": "de_results", "AquaSelect": "aquaselect_results",
}

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
metric_names = ["AURCC ↓", "Cov@95 (%) ↑", "Cov@99 (%) ↑"]
bar_colors = ["#888888", "#4477AA", "#228833", "#EE3377"]

for metric_idx, (ax, metric_name) in enumerate(zip(axes, metric_names)):
    x = np.arange(len(BACKBONES))
    width = 0.18

    for m_idx, method in enumerate(methods):
        values = []
        errors = []
        for backbone_name in BACKBONES:
            rk = metrics_keys[method]
            if method == "DE":
                r = fr[rk][backbone_name]
                if metric_idx == 0: val = r["aurcc"]
                elif metric_idx == 1: val = r["cov95"]
                else: val = r["cov99"]
                values.append(val); errors.append(0)
            else:
                vals = []
                for seed in CFG["seeds"]:
                    r = fr[rk][backbone_name][seed]
                    if metric_idx == 0: vals.append(r["aurcc"])
                    elif metric_idx == 1: vals.append(r["cov95"])
                    else: vals.append(r["cov99"])
                values.append(np.mean(vals)); errors.append(np.std(vals))

        offset = (m_idx - 1.5) * width
        bars = ax.bar(x + offset, values, width, yerr=errors, capsize=3,
                      label=method, color=bar_colors[m_idx], alpha=0.85)

    ax.set_xticks(x)
    ax.set_xticklabels(["ConvNeXt-Tiny", "DeiT-Small"], fontsize=10)
    ax.set_title(metric_name, fontsize=13, fontweight="bold")
    ax.legend(fontsize=8)
    ax.grid(axis="y", alpha=0.2)

plt.suptitle("Cross-Architecture Comparison of Selection Methods",
             fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig(os.path.join(CFG["output_dir"], "cross_architecture_comparison.png"),
            dpi=150, bbox_inches="tight")
plt.show()
print("Saved: cross_architecture_comparison.png")

In [ ]:
# Cell 18
# Statistical significance 

from scipy import stats

print("="*70)
print("  Statistical Comparison: AquaSelect vs SR (per-seed AURCC)")
print("="*70)

for backbone_name in BACKBONES:
    sr_aurccs = [frozen_results["sr_results"][backbone_name][s]["aurcc"] for s in CFG["seeds"]]
    aq_aurccs = [frozen_results["aquaselect_results"][backbone_name][s]["aurcc"] for s in CFG["seeds"]]

    # Per-seed comparison
    print(f"\n  {backbone_name}:")
    for seed, sr_a, aq_a in zip(CFG["seeds"], sr_aurccs, aq_aurccs):
        delta = aq_a - sr_a
        print(f"    Seed {seed}: SR={sr_a:.4f}  AquaSelect={aq_a:.4f}  Δ={delta:+.4f} "
              f"({'better' if delta < 0 else 'worse'})")

    all_better = all(aq < sr for aq, sr in zip(aq_aurccs, sr_aurccs))
    print(f"    AquaSelect better on all seeds: {all_better}")

    # Paired differences
    diffs = [aq - sr for aq, sr in zip(aq_aurccs, sr_aurccs)]
    mean_diff = np.mean(diffs)
    print(f"    Mean AURCC improvement: {mean_diff:+.4f}")

    if len(set(diffs)) > 1:
        stat, p_val = stats.wilcoxon(sr_aurccs, aq_aurccs, alternative="greater")
        print(f"    Wilcoxon signed-rank: statistic={stat:.1f}, p={p_val:.4f}")
    else:
        print(f"    Wilcoxon: cannot compute (identical differences)")

    print(f"    Note: With n=3 seeds, Wilcoxon has limited power. "
          f"Consistent per-seed improvement is the stronger evidence.")

In [ ]:
#cell 19
import time

# Load one ConvNeXt-Tiny model
model = load_trained_model("convnext_tiny", 42).to(device).eval()
dummy = torch.randn(1, 3, 224, 224).to(device)

# Warmup
for _ in range(10):
    with torch.no_grad():
        _ = model(dummy)
torch.cuda.synchronize()

# Single model (AquaSelect)
times = []
for _ in range(100):
    torch.cuda.synchronize()
    t0 = time.perf_counter()
    with torch.no_grad():
        _ = model(dummy)
    torch.cuda.synchronize()
    times.append(time.perf_counter() - t0)

single_ms = np.mean(times) * 1000
single_fps = 1000 / single_ms

# Deep Ensemble (3 models)
models = [load_trained_model("convnext_tiny", s).to(device).eval() for s in [9, 42, 2003]]
times_de = []
for _ in range(100):
    torch.cuda.synchronize()
    t0 = time.perf_counter()
    with torch.no_grad():
        for m in models:
            _ = m(dummy)
    torch.cuda.synchronize()
    times_de.append(time.perf_counter() - t0)

de_ms = np.mean(times_de) * 1000
de_fps = 1000 / de_ms

print(f"AquaSelect (single model): {single_ms:.1f} ms/image = {single_fps:.0f} FPS")
print(f"Deep Ensemble (3 models):  {de_ms:.1f} ms/image = {de_fps:.0f} FPS")
print(f"Speedup: {de_ms/single_ms:.1f}x")

del models, model
torch.cuda.empty_cache()

In [ ]:
# OOD test: pass random/out-of-distribution images through AquaSelect
from PIL import Image
import torchvision.transforms as T

model = load_trained_model("convnext_tiny", 42).to(device).eval()

ood_images = {
    "Random noise": Image.fromarray(np.random.randint(0, 255, (224, 224, 3), dtype=np.uint8)),
    "Solid blue (water)": Image.fromarray(np.full((224, 224, 3), [30, 80, 140], dtype=np.uint8)),
    "Solid green": Image.fromarray(np.full((224, 224, 3), [50, 150, 50], dtype=np.uint8)),
    "Gradient": Image.fromarray(np.tile(np.linspace(0, 255, 224, dtype=np.uint8), (224, 3, 1)).transpose(2, 0, 1).astype(np.uint8)),
    "Checkerboard": Image.fromarray(((np.indices((224, 224)).sum(axis=0) % 2) * 255).astype(np.uint8).repeat(3).reshape(224, 224, 3)),
}

transform = T.Compose([
    T.Resize(256), T.CenterCrop(224), T.ToTensor(),
    T.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

print("OOD Image Analysis (ConvNeXt-Tiny seed 42):")
print(f"{'Image':<20} | {'Max Softmax':>11} | {'Top-1 Pred':>15} | {'Top-2 Pred':>15}")
print("-" * 70)

with torch.no_grad():
    for name, img in ood_images.items():
        tensor = transform(img.convert("RGB")).unsqueeze(0).to(device)
        logits, features = model(tensor)
        probs = F.softmax(logits, dim=1)
        top2 = torch.topk(probs, 2, dim=1)
        max_prob = top2.values[0, 0].item()
        pred1 = label_names[top2.indices[0, 0].item()]
        pred2 = label_names[top2.indices[0, 1].item()]
        print(f"{name:<20} | {max_prob:>10.4f} | {pred1:>15} | {pred2:>15}")

# Compare with typical in-distribution confidence
test_logits = saved_logits["convnext_tiny"]["test_logits"][42]
test_probs = F.softmax(test_logits, dim=1)
id_max_probs = test_probs.max(dim=1).values.numpy()
print(f"\nIn-distribution max softmax: {id_max_probs.mean():.4f} ± {id_max_probs.std():.4f}")
print(f"In-distribution 10th percentile: {np.percentile(id_max_probs, 10):.4f}")
print(f"OOD images would be flagged for review (abstained) at any reasonable threshold.")

del model
torch.cuda.empty_cache()

In [ ]:
# Cell 20
# List all outputs 

output_dir = CFG["output_dir"]
print(f"All outputs in {output_dir}:")
for f in sorted(os.listdir(output_dir)):
    fpath = os.path.join(output_dir, f)
    if os.path.isfile(fpath):
        print(f"  {f} ({os.path.getsize(fpath) / 1e6:.1f} MB)")

In [ ]:
# Cell 21
# Done 
print("Notebook 3 complete.")